<a href="https://colab.research.google.com/github/ashitasingh1230-commits/IT_support_ticket_analysis/blob/main/02_user_directory_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2: User Directory Join (Exploratory)

Goal: enrich ticket data with official department/user info from user_directory.csv.

Finding: tickets contain no clean username/email field. The only identifying
info appears occasionally inside free-text correspondence messages, which
would need regex extraction with uncertain coverage. Scoped out in favor of
the existing user_group_category field, which already provides department-level
insight from the ticket data directly.

# Load both files into pandas





In [9]:
import pandas as pd

tickets_df = pd.read_csv('cleaned_tickets.csv')
users_df = pd.read_csv('users_directory.csv')

print(tickets_df.shape)
print(users_df.shape)

(745, 23)
(769, 3)


# Peek at users_df

In [11]:
users_df.head()

,username,display_name,email
0,a.hussain,Amir Hussain,a.hussain@corplabs.com
1,abrandt,Andrea Brandt,abrandt@corplabs.com
2,acooper,A. Cooper,acooper@corplabs.com
3,ahaddad,Amir Haddad,ahaddad@corplabs.com
4,akapoor,Anil Kapoor,akapoor@corplabs.com


# Check tickets_df's columns

In [13]:
tickets_df.columns.tolist()

['record_id',
 'record_type',
 'ticket',
 'status',
 'correspondence',
 'diagnostics',
 'root_cause',
 'resolution',
 'ticket_parsed',
 'priority',
 'sla_plan',
 'sla_plan_clean',
 'sla_category',
 'root_cause_lower',
 'root_cause_category',
 'os',
 'platform',
 'region',
 'user_group',
 'os_category',
 'region_category',
 'platform_category',
 'user_group_category']

# See all the keys inside one parsed ticket

In [14]:
import json
sample=json.loads(tickets_df['ticket'].iloc[0])
sample.keys()

dict_keys(['submitted_at', 'submitted_title', 'submitted_description', 'priority', 'sla_plan', 'environment', 'applications'])

# Check the applications field

In [15]:
sample['applications']

['Password Reset Portal', 'SSO Portal']

# Check the correspondence field

In [16]:
tickets_df['correspondence'].iloc[0]

'[{"turn_id": 1, "role": "agent", "event_type": "diagnostic_question", "message": "Hi Monica, after the password reset, are you still signed into corporate mail or the SSO portal on your iPhone (WORKSTATION-IPHONE-MC), and do you have any saved passwords in the iOS Mail app or Safari that might still be using the old password? I can see your account CN=mcarson,OU=Corp Users,DC=corplabs,DC=internal is currently locked. Could you also confirm whether you\'re connecting from your usual IP address 10.22.47.88 at the Charlotte office? — Kevin Briggs, Identity Support", "timestamp": "2025-11-18T08:00:00Z"}, {"turn_id": 2, "role": "user", "event_type": "user_clarification", "message": "Yes, I use my iPhone for corporate mail and I had the old password saved in the Mail app and Safari browser. The account locked again about two minutes after I finished the reset. I\'m at my desk in the Charlotte office right now and my phone is on the corporate Wi-Fi. My email is monica.carson@corplabs.com if 

# Parse the correspondence JSON

In [17]:
correspondence=json.loads(tickets_df['correspondence'].iloc[0])
correspondence[0]

{'turn_id': 1,
 'role': 'agent',
 'event_type': 'diagnostic_question',
 'message': "Hi Monica, after the password reset, are you still signed into corporate mail or the SSO portal on your iPhone (WORKSTATION-IPHONE-MC), and do you have any saved passwords in the iOS Mail app or Safari that might still be using the old password? I can see your account CN=mcarson,OU=Corp Users,DC=corplabs,DC=internal is currently locked. Could you also confirm whether you're connecting from your usual IP address 10.22.47.88 at the Charlotte office? — Kevin Briggs, Identity Support",
 'timestamp': '2025-11-18T08:00:00Z'}